# CTNSG Master Kaggle Training Notebook
This notebook orchestrates the end-to-end training pipeline for the Canonical Tractable Neuro-Symbolic Generation Framework.
It assumes execution on Kaggle with dual T4 or P100 GPUs (13-16GB VRAM).

**Note:** This notebook is configured for full training.

In [ ]:
!git clone https://github.com/Borisz42/CTNSG.git
%cd CTNSG
%pip install -r requirements.txt

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim

# Ensure local modules are reachable
sys.path.append(os.path.abspath('.'))

from macroplanner.gvt.model import GraphVQTransformer
from macroplanner.reldit.model import RelDiT
from orchestrator.arbor.planner import ArborPlanner
from realizer.realizer import CTNSGRealizer
from contracts.graph_schema import DiscourseGraph, SemanticNode, SemanticEdge

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on {device}")

## 1. Load Preprocessed Supervisor Datasets (Module 2 & 4)
Loading the true DAGs, SDRT discourse trees, FAAP instruction pairs, and SAIGuard interaction topologies.

In [ ]:
import json
import os
from huggingface_hub import hf_hub_download

def fetch_and_load(filename):
    local_path = f'processed_data/{filename}'
    if not os.path.exists(local_path):
        print(f'Downloading {filename} from Hugging Face...')
        try:
            os.makedirs('processed_data', exist_ok=True)
            downloaded = hf_hub_download(repo_id='Borisz42/CTNSG-Graph-Curriculum', repo_type='dataset', filename=filename)
            import shutil
            shutil.copy(downloaded, local_path)
        except Exception as e:
            print(f'Warning: Could not download {filename}. {e}')
    
    if filename.endswith('.pt'):
        return torch.load(local_path) if os.path.exists(local_path) else []
    elif filename.endswith('.jsonl'):
        data = []
        if os.path.exists(local_path):
            with open(local_path, 'r', encoding='utf-8') as f:
                data = [json.loads(line) for line in f]
        return data

# Load FAAP (Seq2Seq)
faap_data = fetch_and_load('faap_instructions_full.jsonl')
print(f"Loaded {len(faap_data)} FAAP instruction pairs.")

# Load Topological Graphs
curriculum_graphs = fetch_and_load('ctnsg_curriculum.pt')
sdrt_graphs = fetch_and_load('sdrt_graphs_full.pt')
arbor_graphs = fetch_and_load('arbor_graphs_full.pt')
verification_graphs = fetch_and_load('verification_graphs_full.pt')

print(f"Loaded {len(curriculum_graphs)} Curriculum Graphs.")
print(f"Loaded {len(sdrt_graphs)} SDRT Discourse Graphs.")
print(f"Loaded {len(arbor_graphs)} Arbor TDP True DAGs.")
print(f"Loaded {len(verification_graphs)} SAIGuard/Brick Interaction Graphs.")

## 2. Initialize Global Orchestrator
Decoupling semantic and structural intent via Arbor.

In [ ]:
planner = ArborPlanner(input_dim=512, hidden_dim=256).to(device)
global_intent = torch.randn(1, 512).to(device)
decoupled_plan = planner.decouple_plan(global_intent)
print("Decoupled Plan Confidence:", decoupled_plan['confidence'])

## 3. GVT (Graph Vector Tokenization) Training Loop
Compressing continuous topologies into discrete graph tokens.

In [ ]:
print('\n======================================================')
print('Phase 2: Critical Iterative Denoising (CID) Post-Hoc Critic Training')
print('======================================================\n')
print('Freezing the main RelDiT Denoiser parameters...')

# 1. Freeze denoiser, only train critic
for name, param in reldit.named_parameters():
    if 'critic' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Critic only needs a smaller learning rate as it trains on residual logits
critic_optimizer = optim.AdamW(reldit.critic.parameters(), lr=1e-4)
bce_loss = nn.BCELoss()

print(f"[{datetime.now().strftime('%H:%M:%S')}] Starting Critic Training Loop...")
num_critic_epochs = 10
for epoch in range(num_critic_epochs):
    epoch_start = time.time()
    
    # --- Training ---
    reldit.train()
    total_loss_epoch = 0.0
    for graph in train_graphs[:500]: # Train critic on first 500 graphs for speed
        nodes_full = graph['nodes'].to(device)
        max_seq = 1024
        for i in range(0, nodes_full.shape[0], max_seq):
            nodes = nodes_full[i : i + max_seq]
            edges = torch.empty((2,0), dtype=torch.long, device=device)
            
            with torch.no_grad():
                out = gvt(nodes, edges)
                x0 = out['discrete_tokens'][:, 0].unsqueeze(0)
                
            # Sample random timesteps
            t = torch.randint(1, reldit.num_timesteps + 1, (x0.size(0),), device=device)
            
            # Forward Noising
            x_t, mask = reldit.diffusion.add_noise(x0, t)
            
            # Predict Logits
            with torch.no_grad():
                logits = reldit.transformer(x_t)
                probs = torch.softmax(logits, dim=-1)
                
            # Train Critic
            critic_optimizer.zero_grad()
            
            # Residual Logit Trick (CID)
            critic_scores = reldit.get_critic_scores(probs, t)
            
            # Target: ~mask (1.0 for clean, 0.0 for corrupted)
            target = (~mask).float()
            
            loss = bce_loss(critic_scores, target)
            loss.backward()
            critic_optimizer.step()
            total_loss_epoch += loss.item()
            
    avg_train_loss = total_loss_epoch / 500
    
    epoch_duration = time.time() - epoch_start
    timestamp = datetime.now().strftime('%H:%M:%S')
    print(f"[{timestamp}] Epoch {epoch+1}/{num_critic_epochs} | Critic BCE Loss: {avg_train_loss:.4f} | Time: {epoch_duration:.1f}s")

print('\nCritic training complete! The CID framework is now fully integrated.')
print('To generate highly valid graph topologies, ensure use_critic=True is passed to generate()')


In [ ]:
import time
from datetime import datetime, timedelta

gvt = GraphVQTransformer(in_channels=256, hidden_channels=256, num_embeddings=64, num_quantizers=4).to(device)
gvt_optimizer = optim.AdamW(gvt.parameters(), lr=3e-4)

print(f"[{datetime.now().strftime('%H:%M:%S')}] Starting GVT Training Loop (Full Training)...")
for epoch in range(3):
    epoch_start = time.time()
    total_loss_epoch = 0.0
    for graph in curriculum_graphs:
        nodes_full = graph['nodes'].to(device)
        edges_full = graph['edges'].to(device)
        
        # Chunk massive RCM graphs to prevent OOM
        max_seq = 1024
        for i in range(0, nodes_full.shape[0], max_seq):
            nodes = nodes_full[i : i + max_seq]
            edges = torch.empty((2,0), dtype=torch.long, device=device)
            
            gvt_optimizer.zero_grad()
            out = gvt(nodes, edges)
            quantized_latents = out['z_q']
            vq_loss = out['commit_loss']
            
            recon_loss = nn.MSELoss()(quantized_latents, nodes)
            total_loss = recon_loss + vq_loss
            
            total_loss.backward()
            gvt_optimizer.step()
            total_loss_epoch += total_loss.item()
    
    epoch_duration = time.time() - epoch_start
    avg_loss = total_loss_epoch / max(1, len(curriculum_graphs))
    eta_seconds = epoch_duration * (3 - (epoch + 1))
    eta_str = str(timedelta(seconds=int(eta_seconds)))
    
    timestamp = datetime.now().strftime('%H:%M:%S')
    print(f"[{timestamp}] Epoch {epoch+1}/3 | GVT Loss: {avg_loss:.4f} | Time: {epoch_duration:.1f}s | ETA: {eta_str}")

discrete_indices = out['discrete_tokens']
print(f"[{datetime.now().strftime('%H:%M:%S')}] GVT Tokenization complete. Sample discrete indices shape:", discrete_indices.shape)

print("Saving GVT weights before starting RelDiT...")
import os
os.makedirs("ctnsg_export", exist_ok=True)
torch.save(gvt.state_dict(), "ctnsg_export/gvt_weights.pt")


## 4. RelDiT (Relational Diffusion) Training Loop
Training the discrete diffusion model to generate topologies.

In [ ]:
# Data Splitting for Validation
val_split_size = min(200, len(curriculum_graphs) // 10)
train_graphs = curriculum_graphs[:-val_split_size]
val_graphs = curriculum_graphs[-val_split_size:]

print(f"Data split: {len(train_graphs)} training graphs, {len(val_graphs)} validation graphs.")

# RelDiT acts on the discrete indices generated by GVT
reldit = RelDiT(vocab_size=64, d_model=256).to(device)
reldit_optimizer = optim.AdamW(reldit.parameters(), lr=1e-4)

# Early Stopping and Tracking
best_val_loss = float('inf')
patience = 10
patience_counter = 0

# Cache training tokens for Novelty metric
print("Caching training tokens for V.U.N Novelty tracking...")
train_token_cache = set()
with torch.no_grad():
    for graph in train_graphs[:500]: # Cache first 500 for speed
        nodes_full = graph['nodes'].to(device)
        max_seq = 1024
        for i in range(0, nodes_full.shape[0], max_seq):
            nodes = nodes_full[i : i + max_seq]
            edges = torch.empty((2,0), dtype=torch.long, device=device)
            out = gvt(nodes, edges)
            tokens = out['discrete_tokens'][:, 0].cpu().numpy().tobytes()
            train_token_cache.add(tokens)

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Starting RelDiT Training Loop (Full Training)...")
num_epochs = 100
reldit_start_time = time.time()
max_reldit_time = 11 * 3600  # 11 hours limit
for epoch in range(num_epochs):
    epoch_start = time.time()
    
    # --- Training ---

    if time.time() - reldit_start_time > max_reldit_time:
        print(f"\n[{datetime.now().strftime('%H:%M:%S')}] 11-hour time limit reached! Halting RelDiT training safely to preserve Kaggle session.")
        reldit.load_state_dict(torch.load('best_reldit_weights.pt'))
        break
    reldit.train()
    total_loss_epoch = 0.0
    for graph in train_graphs:
        nodes_full = graph['nodes'].to(device)
        max_seq = 1024
        for i in range(0, nodes_full.shape[0], max_seq):
            nodes = nodes_full[i : i + max_seq]
            edges = torch.empty((2,0), dtype=torch.long, device=device)
            
            with torch.no_grad():
                out = gvt(nodes, edges)
                tokens = out['discrete_tokens'][:, 0].unsqueeze(0)
                
            reldit_optimizer.zero_grad()
            loss = reldit(tokens)
            loss.backward()
            reldit_optimizer.step()
            total_loss_epoch += loss.item()
            
    avg_train_loss = total_loss_epoch / max(1, len(train_graphs))
    
    # --- Validation ---
    reldit.eval()
    val_loss_epoch = 0.0
    with torch.no_grad():
        for graph in val_graphs:
            nodes_full = graph['nodes'].to(device)
            max_seq = 1024
            for i in range(0, nodes_full.shape[0], max_seq):
                nodes = nodes_full[i : i + max_seq]
                edges = torch.empty((2,0), dtype=torch.long, device=device)
                out = gvt(nodes, edges)
                tokens = out['discrete_tokens'][:, 0].unsqueeze(0)
                loss = reldit(tokens)
                val_loss_epoch += loss.item()
                
    avg_val_loss = val_loss_epoch / max(1, len(val_graphs))
    
    # --- ETA Calculation ---
    epoch_duration = time.time() - epoch_start
    eta_seconds = epoch_duration * (num_epochs - (epoch + 1))
    eta_str = str(timedelta(seconds=int(eta_seconds)))
    
    timestamp = datetime.now().strftime('%H:%M:%S')
    print(f"[{timestamp}] Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Time: {epoch_duration:.1f}s | ETA: {eta_str}")
    
    # --- Early Stopping Logic ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(reldit.state_dict(), 'best_reldit_weights.pt')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Early stopping triggered after {epoch+1} epochs! Best Val Loss: {best_val_loss:.4f}")
            reldit.load_state_dict(torch.load('best_reldit_weights.pt'))
            break
            
    # --- V.U.N Metrics Evaluation ---
    if (epoch + 1) % 5 == 0:
        print(f"  -> Evaluating V.U.N Metrics (Epoch {epoch+1})...")
        N_samples = 10
        gen_seq_len = 1024
        
        with torch.no_grad():
            gen_tokens = reldit.generate(batch_size=N_samples, seq_len=gen_seq_len, device=device, use_critic=False)
            
            # Validity: Average Critic Confidence
            scaled_logits = reldit.transformer(gen_tokens) / 1.0
            probs = torch.softmax(scaled_logits, dim=-1)
            # Validity (pre-phase 2 proxy): Mean token probability
            pred_probs = torch.gather(probs, -1, gen_tokens.unsqueeze(-1)).squeeze(-1)
            validity = pred_probs.mean().item() * 100.0
            
            # Uniqueness: Number of unique generated sequences
            unique_seqs = set([tuple(seq.tolist()) for seq in gen_tokens])
            uniqueness = (len(unique_seqs) / N_samples) * 100.0
            
            # Novelty: How many generated sequences are NOT in the training cache
            novel_count = 0
            for seq in unique_seqs:
                seq_bytes = torch.tensor(seq, dtype=torch.long).numpy().tobytes()
                if seq_bytes not in train_token_cache:
                    novel_count += 1
            novelty = (novel_count / max(1, len(unique_seqs))) * 100.0
            
        print(f"  -> [V.U.N] Validity: {validity:.1f}% | Uniqueness: {uniqueness:.1f}% | Novelty: {novelty:.1f}%")


## 5. Realizer Inference Pipeline
Integrating the VNPool, O(1) GREATGRAMMA enforcement, and SafeLLM to generate text.

In [ ]:
print("\nInitializing Realizer Pipeline...")
realizer = CTNSGRealizer()

# Create a stub discourse graph to feed into the Realizer
inference_graph = DiscourseGraph(
    graph_id="infer_001",
    nodes=[
        SemanticNode(node_id="n1", concept="System", vq_index=12),
        SemanticNode(node_id="n2", concept="Macroplanner", vq_index=45)
    ],
    edges=[
        SemanticEdge(source_id="n1", target_id="n2", relation_type="triggers")
    ]
)

schema = {"type": "object", "properties": {"output": {"type": "string"}}}
result = realizer.generate(inference_graph, schema, context_lines=5)

print("\n=== Realizer Final Output ===")
print(result)

## 6. Model Weight Export & Distribution
Saves the trained components (GVT and RelDiT) to /kaggle/working/ and packages them into a .zip file. This zip file can be easily downloaded or natively linked as a 'Notebook Output' dataset into the Evaluation Suite notebook without needing manual downloads.

In [ ]:
import shutil

print('\nSaving trained models...')
os.makedirs('ctnsg_export', exist_ok=True)
torch.save(gvt.state_dict(), 'ctnsg_export/gvt_weights.pt')
torch.save(reldit.state_dict(), 'ctnsg_export/reldit_weights.pt')

# Create an export manifest
manifest = {
    'architecture': 'CTNSG',
    'modules': ['GVT', 'RelDiT'],
    'dim': 256
}
with open('ctnsg_export/manifest.json', 'w') as f:
    json.dump(manifest, f, indent=4)

# Zip the export directory
print('Packaging into ctnsg_release.zip...')
shutil.make_archive('/kaggle/working/ctnsg_release', 'zip', 'ctnsg_export')

print('Export complete! ctnsg_release.zip is now available in the Kaggle /kaggle/working/ directory.')